# LokerLens — Pelatihan Model Klasifikasi Lowongan Kerja (Asli vs Palsu)

**Sprint 1 · Fitur 2: Pelatihan Model Klasifikasi** · Kelompok LokerLens (Pembangunan Perangkat Lunak I4)

Alur notebook:
1. Konfigurasi & memuat data (**palsu = Kaggle**, **asli = hasil scraping**)
2. Penyatuan, pembersihan, dan **cek kebocoran bahasa** (penting, lihat catatan di bagian 3)
3. Preprocessing teks + TF-IDF
4. Perbandingan 4 model: Naive Bayes, Logistic Regression, SVM, Random Forest
5. Evaluasi model terbaik (test set) + analisis per bahasa
6. Fungsi `periksa()` (kategori, confidence score, kata pemicu, indikator) untuk dipakai back end
7. Ekspor model

> Output sistem = **alat bantu keputusan**, bukan penetapan penipuan (BR-04 di SRS).

## 1. Setup & Konfigurasi

In [ ]:
# Jalankan sekali kalau library belum ada (hapus tanda # di bawah)
# !pip install -q pandas scikit-learn matplotlib seaborn joblib Sastrawi deep-translator

In [ ]:
import os, re, json, warnings, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, precision_recall_fscore_support)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42

In [ ]:
# ===================== KONFIGURASI (ubah di sini) =====================
PATH_KAGGLE = "fake_job_postings.csv"       # dataset Kaggle EMSCAD (kolom: title, description, requirements, ..., fraudulent)
PATH_ASLI   = "loker_asli_scraped.csv"      # hasil scraping JobStreet dkk (skema di bawah)

# Skema CSV hasil scraping (lowongan ASLI), minimal kolom ini:
#   nama_perusahaan, posisi, deskripsi, info_kontak (boleh kosong), sumber (mis. jobstreet)

USE_DEMO_IF_MISSING = True    # True: kalau file di atas belum ada, pakai data DEMO sintetis (cuma buat ngetes notebook jalan)
MIN_CHARS           = 50      # sesuai form: deskripsi minimal 50 karakter
TRANSLATE_PALSU     = False   # True: terjemahkan lowongan palsu Kaggle (Inggris) ke Indonesia (lihat bagian 3)
TRANSLATE_LIMIT     = 400     # batasi jumlah yang diterjemahkan biar tidak kelamaan
USE_STEMMING        = False   # True: stemming Sastrawi (lebih lambat, kadang membantu)
AMBANG_PRIORITAS    = 0.80    # BR-05 (TBD di SRS) -> usulan awal, sepakati dengan tim
TEST_SIZE           = 0.20
# ======================================================================

## 2. Modul inference (`lokerlens_infer.py`)

Preprocessing, kata pemicu, dan indikator ditulis sebagai **file modul terpisah**. Alasannya: model `.joblib` tidak boleh bergantung pada fungsi yang didefinisikan di notebook (bakal error saat di-load di back end). Dengan modul ini, **training dan back end memakai preprocessing yang persis sama**.

In [ ]:
%%writefile lokerlens_infer.py
"""Modul inference LokerLens: preprocessing, prediksi, kata pemicu, indikator rule-based."""
import re
import numpy as np
import joblib

USE_STEMMING = False   # harus sama dengan saat training (disimpan juga di metadata)

# ---------- stopword ----------
try:
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    _SW_ID = set(StopWordRemoverFactory().get_stop_words())
except Exception:
    _SW_ID = set("yang dan di ke dari untuk dengan pada ini itu atau adalah dalam akan juga oleh sebagai kami kita anda dapat bisa serta para atas".split())
_SW_EN = set("the a an and or of to in for on with is are be as at by from this that it we you your our will can".split())
STOPWORDS = _SW_ID | _SW_EN

_stemmer = None
def _stem(tokens):
    global _stemmer
    if _stemmer is None:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        _stemmer = StemmerFactory().create_stemmer()
    return [_stemmer.stem(t) for t in tokens]

# ---------- regex ----------
HTML_RE    = re.compile(r"<[^>]+>")
PLACEHOLDER_RE = re.compile(r"#(?:url|email|phone)_[0-9a-f]+#", re.I)   # artefak anonimisasi EMSCAD (bisa jadi kebocoran label!)
URL_RE     = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE   = re.compile(r"[\w.+-]+@([\w-]+(?:\.[\w-]+)+)")
WA_RE      = re.compile(r"(?<!\d)(?:\+?62|0)8\d[\d\s-]{7,12}")
RP_RE      = re.compile(r"\brp\.?\s?\d[\d.,]*(?:\s?(?:juta|jt|ribu|rb|k)\b)?", re.I)
FREE_MAIL  = ("gmail.", "yahoo.", "ymail.", "hotmail.", "outlook.", "live.", "rocketmail.")

def _email_token(m):
    dom = m.group(1).lower()
    return " emailgratisan " if dom.startswith(FREE_MAIL) else " emailresmi "

def preprocess(text: str) -> str:
    t = str(text if text is not None else "")
    t = HTML_RE.sub(" ", t)
    t = PLACEHOLDER_RE.sub(" ", t)
    t = t.lower()
    t = EMAIL_RE.sub(_email_token, t)
    t = URL_RE.sub(" tokenurl ", t)
    t = WA_RE.sub(" nomorwa ", t)
    t = RP_RE.sub(" nominalrp ", t)
    t = re.sub(r"\d+", " ", t)
    t = re.sub(r"[^a-z\s]", " ", t)
    tokens = [w for w in t.split() if len(w) > 1 and w not in STOPWORDS]
    if USE_STEMMING:
        tokens = _stem(tokens)
    return " ".join(tokens)

def gabung_teks(nama_perusahaan="", posisi="", deskripsi="", info_tambahan="") -> str:
    return " . ".join(str(x) for x in [nama_perusahaan, posisi, deskripsi, info_tambahan] if x is not None and str(x).strip())

# ---------- indikator rule-based (selaras mockup: biaya, gaji tidak wajar, email gratisan) ----------
def _nominal_rp(text):
    vals = []
    for m in RP_RE.finditer(text.lower()):
        s = m.group(0)
        num = re.search(r"\d[\d.,]*", s).group(0).replace(".", "").replace(",", "")
        if not num.isdigit():
            continue
        v = int(num)
        if re.search(r"juta|jt", s): v *= 1_000_000
        elif re.search(r"ribu|rb|k\b", s): v *= 1_000
        vals.append(v)
    return vals

def indikator_rule_based(raw_text: str):
    t = raw_text.lower()
    hasil = []
    if re.search(r"(biaya|bayar|membayar|transfer|iuran|deposit)[^.]{0,60}(administrasi|pendaftaran|seragam|tes|pelatihan|training|wawancara|interview|registrasi)", t) \
       or re.search(r"(administrasi|pendaftaran|seragam|registrasi)[^.]{0,30}(biaya|bayar|transfer|rp)", t):
        hasil.append("Indikasi biaya administrasi/pendaftaran (permintaan pembayaran)")
    if any(d in e.lower() for e in re.findall(r"[\w.+-]+@[\w.-]+", t) for d in FREE_MAIL):
        hasil.append("Domain email tidak resmi (email gratisan)")
    gaji = [v for v in _nominal_rp(t) if v >= 5_000_000]
    if gaji and re.search(r"tanpa pengalaman|fresh ?graduate|tanpa syarat|tidak perlu pengalaman|pengalaman tidak diperlukan|lulusan (sma|smk)", t):
        hasil.append("Iming-iming gaji tidak wajar tanpa kualifikasi spesifik")
    if re.search(r"(wawancara|interview)[^.]{0,40}(hotel|penginapan|kafe|cafe|rumah|apartemen)", t):
        hasil.append("Lokasi wawancara mencurigakan (di luar kantor)")
    return hasil

# ---------- kata pemicu ----------
def _bobot_fitur(clf):
    """Bobot per fitur ke arah kelas PALSU (positif = makin mencurigakan)."""
    if hasattr(clf, "calibrated_classifiers_"):          # SVM terkalibrasi
        ws = []
        for c in clf.calibrated_classifiers_:
            est = getattr(c, "estimator", None) or getattr(c, "base_estimator", None)
            ws.append(est.coef_[0])
        return np.mean(ws, axis=0)
    if hasattr(clf, "coef_"):                            # LR / linear SVM
        return clf.coef_[0]
    if hasattr(clf, "feature_log_prob_"):                # Naive Bayes
        return clf.feature_log_prob_[1] - clf.feature_log_prob_[0]
    if hasattr(clf, "feature_importances_"):             # Random Forest (tidak berarah, hanya kepentingan)
        return clf.feature_importances_
    raise ValueError("Model tidak didukung untuk kata pemicu")

def kata_pemicu(pipe, teks_bersih: str, top_n=5):
    vec, clf = pipe.named_steps["tfidf"], pipe.named_steps["clf"]
    x = vec.transform([teks_bersih])
    kontrib = x.multiply(_bobot_fitur(clf)).tocoo()
    fitur = vec.get_feature_names_out()
    pasang = sorted(zip(kontrib.col, kontrib.data), key=lambda p: -p[1])
    kandidat = [fitur[i] for i, v in pasang[: top_n * 3] if v > 0]
    # buang unigram yang sudah tercakup bigram yang terpilih
    hasil = []
    for k in kandidat:
        if any((k != o and k in o.split()) for o in kandidat):
            continue
        hasil.append(k)
    return hasil[:top_n]

# ---------- API utama untuk back end ----------
def muat_model(path="model_lokerlens.joblib"):
    return joblib.load(path)

def periksa(pipe, nama_perusahaan="", posisi="", deskripsi="", info_tambahan="", ambang_prioritas=0.80):
    raw = gabung_teks(nama_perusahaan, posisi, deskripsi, info_tambahan)
    bersih = preprocess(raw)
    p = float(pipe.predict_proba([bersih])[0][1])          # probabilitas PALSU
    if p >= ambang_prioritas:
        risiko, kategori = "tinggi", "Lowongan berpotensi penipuan"
    elif p >= 0.5:
        risiko, kategori = "sedang", "Lowongan berpotensi penipuan"
    else:
        risiko, kategori = "rendah", "Lowongan kemungkinan asli"
    return {
        "kategori": kategori,
        "tingkat_risiko": risiko,
        "confidence_score": round(p * 100, 1),              # % probabilitas penipuan
        "prioritas_tinggi": p >= ambang_prioritas,
        "kata_pemicu": kata_pemicu(pipe, bersih),
        "indikator_rule_based": indikator_rule_based(raw),
    }

In [ ]:
import importlib, lokerlens_infer
lokerlens_infer.USE_STEMMING = USE_STEMMING
importlib.reload(lokerlens_infer)
lokerlens_infer.USE_STEMMING = USE_STEMMING   # reload mereset nilai, set ulang
from lokerlens_infer import preprocess, gabung_teks, periksa, kata_pemicu, indikator_rule_based

print(preprocess("PT Contoh membuka lowongan. Bayar biaya administrasi Rp150.000 ke 0812-3456-7890, email: hrd@gmail.com"))

## 3. Memuat & Menyatukan Data

**Rencana:** palsu dari Kaggle (EMSCAD, `fraudulent == 1`), asli dari scraping.

> ⚠️ **Risiko kebocoran bahasa/gaya (data leakage).** EMSCAD berbahasa **Inggris** dan lowongan hasil scraping berbahasa **Indonesia**. Kalau digabung begitu saja, model bisa "belajar" *bahasa = label* (Inggris → palsu, Indonesia → asli) dan akurasinya terlihat 99% padahal tidak bisa membedakan penipuan di data nyata (yang berbahasa Indonesia). Notebook ini menyediakan:
> - **Cek otomatis** proporsi bahasa per kelas (bagian 3.3),
> - Opsi **terjemahan** palsu → Indonesia (`TRANSLATE_PALSU`),
> - **Evaluasi per bahasa** di bagian 6.
>
> Solusi terbaik: tambah lowongan palsu **berbahasa Indonesia** (laporan penipuan, postingan peringatan di media sosial) dan/atau lowongan asli berbahasa Inggris dari EMSCAD (`label==0`), supaya tiap kelas punya dua bahasa.

In [ ]:
def load_kaggle_palsu(path):
    df = pd.read_csv(path)
    df = df[df["fraudulent"] == 1].copy()
    gabung = lambda *cols: df[list(cols)].fillna("").astype(str).agg(" ".join, axis=1)
    out = pd.DataFrame({
        "nama_perusahaan": df["company_profile"].fillna("").astype(str).str[:200],   # EMSCAD tidak punya nama perusahaan
        "posisi": df["title"].fillna("").astype(str),
        "deskripsi": gabung("description", "requirements", "benefits"),
        "info_kontak": "",
        "sumber": "kaggle_emscad",
        "label": 1,
    })
    return out

def load_scraping_asli(path):
    df = pd.read_csv(path)
    for c in ["nama_perusahaan", "posisi", "deskripsi", "info_kontak", "sumber"]:
        if c not in df.columns:
            df[c] = ""
    df = df[["nama_perusahaan", "posisi", "deskripsi", "info_kontak", "sumber"]].copy()
    df["label"] = 0
    return df

def buat_data_demo(n=300, seed=SEED):
    # DATA SINTETIS: cuma untuk memastikan pipeline jalan. JANGAN dipakai untuk laporan.
    rng = np.random.default_rng(seed)
    pos = ["Staff Administrasi", "Admin Gudang", "Customer Service", "Operator Produksi", "Akuntan Junior", "Marketing Executive"]
    asli_t = ["Dibutuhkan {p} dengan pengalaman minimal {x} tahun. Kualifikasi S1, mampu bekerja dalam tim, menguasai Microsoft Office. Gaji sesuai UMR dan BPJS. Kirim CV ke email resmi perusahaan.",
              "Kami membuka lowongan {p}. Persyaratan pendidikan D3/S1, jujur, teliti, dan komunikatif. Wawancara dilakukan di kantor pusat. Benefit tunjangan makan dan transport.",
              "Lowongan {p} penempatan Surabaya. Pengalaman {x} tahun di bidang terkait, bersedia lembur, jenjang karier jelas, proses seleksi melalui tes tertulis dan wawancara."]
    palsu_t = ["Lowongan {p} gaji Rp10.000.000 per bulan tanpa pengalaman! Wajib transfer biaya administrasi Rp150.000 sebelum wawancara. Hubungi WhatsApp 081234567890 atau email hrd{x}@gmail.com",
               "Dibutuhkan segera {p}, fresh graduate lulusan SMA, gaji Rp8 juta. Bayar biaya pendaftaran dan seragam dulu, wawancara di hotel. Proses cepat langsung diterima, kirim data ke wa 0857{x}123456",
               "Kerja dari rumah {p} penghasilan Rp12.000.000 tanpa syarat khusus, daftar sekarang transfer deposit Rp200.000, kuota terbatas. Info lengkap hubungi 0812{x}0099 email lowongan{x}@yahoo.com"]
    def mk(templates, label, sumber):
        rows = []
        for i in range(n):
            p = rng.choice(pos); x = int(rng.integers(1, 6)) * 111
            rows.append({"nama_perusahaan": f"PT Demo {label}{i}", "posisi": p,
                         "deskripsi": rng.choice(templates).format(p=p, x=x % 6 + 1 if label == 0 else x),
                         "info_kontak": "", "sumber": sumber, "label": label})
        return pd.DataFrame(rows)
    return mk(asli_t, 0, "demo_asli"), mk(palsu_t, 1, "demo_palsu")

pakai_demo = False
if os.path.exists(PATH_KAGGLE) and os.path.exists(PATH_ASLI):
    df_palsu = load_kaggle_palsu(PATH_KAGGLE)
    df_asli  = load_scraping_asli(PATH_ASLI)
elif USE_DEMO_IF_MISSING:
    print("⚠️  File dataset belum ada -> pakai DATA DEMO SINTETIS (hasil tidak bermakna, hanya tes pipeline).")
    df_asli, df_palsu = buat_data_demo()
    pakai_demo = True
else:
    raise FileNotFoundError("Letakkan fake_job_postings.csv dan loker_asli_scraped.csv di folder yang sama dengan notebook.")

print("Palsu:", len(df_palsu), "| Asli:", len(df_asli))

### 3.1 Terjemahan opsional (palsu Kaggle → Indonesia)

In [ ]:
def terjemahkan(teks, tujuan="id"):
    from deep_translator import GoogleTranslator
    gt = GoogleTranslator(source="auto", target=tujuan)
    potongan = [teks[i:i+4500] for i in range(0, len(teks), 4500)]
    return " ".join(gt.translate(p) for p in potongan)

if TRANSLATE_PALSU and not pakai_demo:
    idx = df_palsu.sample(min(TRANSLATE_LIMIT, len(df_palsu)), random_state=SEED).index
    df_palsu = df_palsu.loc[idx].copy()          # hanya pakai yang diterjemahkan -> semua palsu jadi Indonesia
    for kol in ["posisi", "deskripsi"]:
        df_palsu[kol] = df_palsu[kol].map(lambda s: terjemahkan(s[:9000]) if s.strip() else s)
    df_palsu["nama_perusahaan"] = ""
    print("Terjemahan selesai:", len(df_palsu))
else:
    print("Terjemahan dilewati.")

### 3.2 Gabung, bersihkan, deduplikasi

In [ ]:
df = pd.concat([df_asli, df_palsu], ignore_index=True)
df["teks_mentah"] = [gabung_teks(a, b, c, d) for a, b, c, d in
                     zip(df.nama_perusahaan, df.posisi, df.deskripsi, df.info_kontak)]
df["teks_bersih"] = df["teks_mentah"].map(preprocess)

n0 = len(df)
df = df[df["deskripsi"].fillna("").str.len() >= MIN_CHARS]                 # sesuai validasi form
df = df[df["teks_bersih"].str.split().str.len() >= 5]                      # buang yang hampir kosong setelah preprocessing
df = df.drop_duplicates(subset="teks_bersih").reset_index(drop=True)      # EMSCAD banyak duplikat
print(f"{n0} -> {len(df)} baris setelah pembersihan")
print(df["label"].value_counts().rename({0: "asli", 1: "palsu"}))

### 3.3 Cek kebocoran bahasa

In [ ]:
ID_HINT = set("yang dan untuk dengan dari pada kami anda akan dapat sebagai dalam atau adalah gaji lowongan perusahaan".split())
EN_HINT = set("the and for with you our will are your work job team experience company".split())

def deteksi_bahasa(t):
    w = re.findall(r"[a-z]+", t.lower())
    i, e = sum(x in ID_HINT for x in w), sum(x in EN_HINT for x in w)
    return "id" if i >= e else "en"

df["bahasa"] = df["teks_mentah"].map(deteksi_bahasa)
tab = pd.crosstab(df["label"].map({0: "asli", 1: "palsu"}), df["bahasa"])
print(tab)

terkonfound = (tab.gt(0).sum(axis=1) < 2).any()
if terkonfound and not pakai_demo:
    print("\n🚨 PERINGATAN: ada kelas yang hanya punya SATU bahasa. Skor evaluasi akan terlalu bagus (kebocoran bahasa).")
    print("   Tambah data palsu berbahasa Indonesia / asli berbahasa Inggris, atau aktifkan TRANSLATE_PALSU.")
elif pakai_demo:
    print("\n(Data demo: abaikan.)")
else:
    print("\n✅ Tiap kelas punya lebih dari satu bahasa.")

### 3.4 Eksplorasi singkat

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df["label"].map({0: "asli", 1: "palsu"}).value_counts().plot.bar(ax=ax[0], color=["#4c72b0", "#c44e52"], rot=0)
ax[0].set_title("Distribusi kelas")
df.assign(panjang=df["teks_bersih"].str.split().str.len(), kelas=df["label"].map({0: "asli", 1: "palsu"})) \
  .pipe(lambda d: sns.histplot(data=d, x="panjang", hue="kelas", bins=40, ax=ax[1]))
ax[1].set_title("Panjang teks (jumlah token)"); ax[1].set_xlim(0, df["teks_bersih"].str.split().str.len().quantile(0.99))
plt.tight_layout(); plt.show()

## 4. Split Data

In [ ]:
X = df["teks_bersih"]; y = df["label"]
strata = df["label"].astype(str) + "_" + df["bahasa"]           # jaga proporsi kelas x bahasa
if strata.value_counts().min() < 2:
    strata = y
X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(
    X, y, df.index, test_size=TEST_SIZE, stratify=strata, random_state=SEED)
print("Train:", len(X_tr), "| Test:", len(X_te))
print("Rasio palsu di train:", round(y_tr.mean(), 3), "| di test:", round(y_te.mean(), 3))

## 5. Perbandingan 4 Model (Cross-Validation 5-fold pada data train)

- TF-IDF unigram+bigram (`max_features=20000`, `sublinear_tf=True`).
- `class_weight='balanced'` untuk LR, SVM, RF (Naive Bayes tidak mendukungnya; ia memakai prior kelas dari data).
- SVM dibungkus `CalibratedClassifierCV` agar punya `predict_proba` (dibutuhkan untuk confidence score).
- Metrik utama: **F1 & recall kelas palsu** (kita lebih takut lowongan palsu lolos dibanding salah tandai).

In [ ]:
def buat_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, max_features=20000, sublinear_tf=True)

kandidat = {
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0, random_state=SEED),
    "SVM (linear)": CalibratedClassifierCV(LinearSVC(class_weight="balanced", random_state=SEED), cv=3),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=SEED),
}

skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
metrik = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1", "roc_auc": "roc_auc"}

baris = []
for nama, clf in kandidat.items():
    pipe = Pipeline([("tfidf", buat_tfidf()), ("clf", clf)])
    s = cross_validate(pipe, X_tr, y_tr, cv=skf, scoring=metrik, n_jobs=-1)
    baris.append({"model": nama, **{k: s[f"test_{k}"].mean() for k in metrik},
                  "f1_std": s["test_f1"].std(), "waktu_fit_s": s["fit_time"].mean()})
    print(f"✓ {nama}")

hasil_cv = pd.DataFrame(baris).set_index("model").sort_values(["f1", "recall"], ascending=False)
hasil_cv.round(4)

In [ ]:
ax = hasil_cv[["accuracy", "precision", "recall", "f1", "roc_auc"]].plot.bar(figsize=(10, 4.5), rot=15)
ax.set_ylim(max(0, hasil_cv[["accuracy","precision","recall","f1","roc_auc"]].min().min() - 0.05), 1.01)
ax.set_title("Perbandingan model (rata-rata CV 5-fold, kelas palsu)"); ax.legend(loc="lower right", ncol=5)
plt.tight_layout(); plt.show()

model_terbaik = hasil_cv.index[0]
print("Model terbaik berdasarkan F1 (tie-break recall):", model_terbaik)

> **Pilih model:** default notebook mengambil yang F1-nya tertinggi. Kalau selisih F1 antar model **sangat tipis** (di bawah std fold), pilih yang **bisa dijelaskan** (Logistic Regression / SVM linear) karena kata pemicu jadi lebih bermakna. Ubah manual di sel berikut bila perlu.

In [ ]:
MODEL_FINAL = model_terbaik          # <- ganti manual, mis. "Logistic Regression", kalau mau
print("Model final:", MODEL_FINAL)

## 6. Evaluasi pada Test Set

In [ ]:
from sklearn.base import clone
pipe_final = Pipeline([("tfidf", buat_tfidf()), ("clf", clone(kandidat[MODEL_FINAL]))]).fit(X_tr, y_tr)

proba = pipe_final.predict_proba(X_te)[:, 1]
pred = (proba >= 0.5).astype(int)

print(classification_report(y_te, pred, target_names=["asli", "palsu"], digits=4))
auc = roc_auc_score(y_te, proba); print("ROC-AUC:", round(auc, 4))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ConfusionMatrixDisplay(confusion_matrix(y_te, pred), display_labels=["asli", "palsu"]).plot(ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title("Confusion matrix (test)")
fpr, tpr, _ = roc_curve(y_te, proba); ax[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}"); ax[1].plot([0, 1], [0, 1], "--", c="gray")
ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate"); ax[1].set_title("ROC"); ax[1].legend()
plt.tight_layout(); plt.show()

### 6.1 Evaluasi per bahasa (deteksi kebocoran bahasa)

In [ ]:
bhs_te = df.loc[idx_te, "bahasa"].values
rows = []
for b in np.unique(bhs_te):
    m = bhs_te == b
    if m.sum() == 0: continue
    p, r, f, _ = precision_recall_fscore_support(y_te[m], pred[m], average="binary", zero_division=0)
    rows.append({"bahasa": b, "n_test": int(m.sum()), "n_palsu": int(y_te[m].sum()), "precision": p, "recall": r, "f1": f})
pd.DataFrame(rows).round(4)

Kalau performa pada bahasa Indonesia **jauh lebih buruk** dari bahasa Inggris (atau `n_palsu` Indonesia sangat sedikit), itu tanda model belum siap untuk input nyata LokerLens. Tambah data palsu berbahasa Indonesia.

### 6.2 Sensitivitas ambang prioritas tinggi (BR-05)

In [ ]:
rows = []
for th in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    flag = proba >= th
    tp = int((flag & (y_te.values == 1)).sum()); fp = int((flag & (y_te.values == 0)).sum())
    rows.append({"ambang": th, "diprioritaskan": int(flag.sum()), "palsu_tertangkap(recall)": round(tp / max(1, y_te.sum()), 3),
                 "precision": round(tp / max(1, flag.sum()), 3), "salah_tandai(FP)": fp})
pd.DataFrame(rows)

Gunakan tabel ini buat usulan nilai **BR-05** ke tim: ambang makin tinggi → antrean Analis makin pendek & presisi naik, tapi recall turun.

### 6.3 Kata paling berpengaruh (global)

In [ ]:
from lokerlens_infer import _bobot_fitur
vec, clf = pipe_final.named_steps["tfidf"], pipe_final.named_steps["clf"]
w = _bobot_fitur(clf); fn = vec.get_feature_names_out()
urut = np.argsort(w)
if hasattr(clf, "feature_importances_"):
    print("Random Forest: bobot tidak berarah, hanya kepentingan fitur.")
    print(pd.Series(w, fn).sort_values(ascending=False).head(20))
else:
    print("Kata -> PALSU:", [fn[i] for i in urut[::-1][:20]])
    print("Kata -> ASLI :", [fn[i] for i in urut[:20]])

## 7. Latih Ulang pada Seluruh Data & Uji `periksa()`

In [ ]:
# Setelah metrik dilaporkan (dari test set), model produksi dilatih ulang pada SEMUA data.
pipe_prod = Pipeline([("tfidf", buat_tfidf()), ("clf", clone(kandidat[MODEL_FINAL]))]).fit(X, y)

contoh = dict(
    nama_perusahaan="PT Mandiri Sukses Global", posisi="Staff Administrasi",
    deskripsi="PT Mandiri Sukses Global membuka lowongan Staff Administrasi. Gaji Rp8.000.000-Rp12.000.000 per bulan, tanpa pengalaman. "
              "Kandidat diminta membayar biaya administrasi Rp150.000 sebelum wawancara.",
    info_tambahan="Email: mandirisuksesglobal@gmail.com. Kontak rekrutmen melalui WhatsApp.")
print(json.dumps(periksa(pipe_prod, ambang_prioritas=AMBANG_PRIORITAS, **contoh), indent=2, ensure_ascii=False))

In [ ]:
contoh_asli = dict(
    nama_perusahaan="PT Teknologi Nusantara", posisi="Akuntan Junior",
    deskripsi="Dibutuhkan Akuntan Junior, minimal S1 Akuntansi, pengalaman 1 tahun, menguasai Microsoft Excel dan software akuntansi. "
              "Gaji sesuai UMR Surabaya, BPJS, tunjangan makan. Wawancara di kantor kami.",
    info_tambahan="Kirim CV ke hrd@teknologinusantara.co.id")
print(json.dumps(periksa(pipe_prod, ambang_prioritas=AMBANG_PRIORITAS, **contoh_asli), indent=2, ensure_ascii=False))

## 8. Ekspor untuk Back End

In [ ]:
joblib.dump(pipe_prod, "model_lokerlens.joblib", compress=3)
hasil_cv.round(4).to_csv("perbandingan_model_cv.csv")

meta = {
    "model_final": MODEL_FINAL,
    "tanggal_latih": datetime.date.today().isoformat(),
    "sklearn_version": sklearn.__version__,
    "use_stemming": USE_STEMMING,
    "ambang_prioritas_tinggi": AMBANG_PRIORITAS,
    "jumlah_data": {"asli": int((y == 0).sum()), "palsu": int((y == 1).sum())},
    "metrik_test": {
        "precision_palsu": float(precision_recall_fscore_support(y_te, pred, average="binary")[0]),
        "recall_palsu":    float(precision_recall_fscore_support(y_te, pred, average="binary")[1]),
        "f1_palsu":        float(precision_recall_fscore_support(y_te, pred, average="binary")[2]),
        "roc_auc": float(auc),
    },
    "data_demo": bool(pakai_demo),
}
with open("model_lokerlens_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(json.dumps(meta, indent=2, ensure_ascii=False))
print("\nFile untuk back end: model_lokerlens.joblib, lokerlens_infer.py, model_lokerlens_meta.json")

### Cara pakai di back end (kasih ke Gendhis)

```python
# requirements: scikit-learn==<versi di meta json>, joblib, Sastrawi (opsional)
from lokerlens_infer import muat_model, periksa

model = muat_model("model_lokerlens.joblib")      # load sekali saat server start
hasil = periksa(model,
                nama_perusahaan="...", posisi="...", deskripsi="...", info_tambahan="...",
                ambang_prioritas=0.80)             # BR-05
# hasil -> dict: kategori, tingkat_risiko, confidence_score (%), prioritas_tinggi, kata_pemicu[], indikator_rule_based[]
```

**Catatan:** versi `scikit-learn` di server harus sama dengan di `model_lokerlens_meta.json`, dan `USE_STEMMING` harus konsisten.

### Checklist sebelum dikumpulkan
- [ ] Ganti data demo dengan dataset asli (Kaggle + hasil scraping) dan jalankan ulang dari atas
- [ ] Pastikan peringatan kebocoran bahasa di bagian 3.3 sudah ✅
- [ ] Tentukan nilai **BR-05** bareng tim (pakai tabel 6.2) lalu update `AMBANG_PRIORITAS`
- [ ] Salin tabel perbandingan model + confusion matrix ke laporan